# 04 — Correlações Estatísticas

Análise de correlações entre variáveis do projeto Crimes Violentos DF.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

## Rascunho 1 — Correlação Linear (Pearson / Spearman)

Mede a associação linear entre duas variáveis numéricas (ex.: taxa de homicídios vs. taxa de roubos, ou crimes vs. variáveis socioeconômicas por RA).

In [ ]:
from scipy.stats import pearsonr, spearmanr


def correlacao_linear(df, col_x, col_y, method="pearson"):
    """
    Calcula correlação linear entre col_x e col_y.

    Parameters
    ----------
    df : DataFrame com os dados.
    col_x, col_y : nomes das colunas numéricas.
    method : 'pearson' (linear paramétrica) ou 'spearman' (postos, não-paramétrica).

    Returns
    -------
    dict com coeficiente (r), p-valor e n de observações válidas.
    """
    par = df[[col_x, col_y]].dropna()
    x = par[col_x].to_numpy()
    y = par[col_y].to_numpy()

    if len(x) < 3:
        raise ValueError("São necessárias ao menos 3 observações válidas.")

    if method == "pearson":
        r, p = pearsonr(x, y)
    elif method == "spearman":
        r, p = spearmanr(x, y)
    else:
        raise ValueError("method deve ser 'pearson' ou 'spearman'.")

    return {
        "var_x": col_x,
        "var_y": col_y,
        "method": method,
        "coeficiente": round(float(r), 4),
        "p_valor": float(p),
        "n": int(len(x)),
    }


def matriz_correlacao(df, colunas, method="pearson"):
    """Retorna a matriz de correlação entre as colunas numéricas informadas."""
    return df[colunas].corr(method=method)


# Exemplo de uso (substituir pelo caminho real da base analítica):
# df = pd.read_csv("02_base_analitica.csv")
# res = correlacao_linear(df, "taxa_homicidio", "taxa_roubo", method="pearson")
# print(res)
# cols = ["taxa_homicidio", "taxa_roubo", "renda_media", "populacao"]
# print(matriz_correlacao(df, cols))

## Rascunho 2 — Correlação Logística (Regressão Logística)

Modela a PROBABILIDADE de um evento binário (ex.: 'RA tem alto índice de crimes violentos' = 1/0) em função de variáveis explicativas (renda, desigualdade, escolaridade etc.).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score


def correlacao_logistica(df, colunas_explicativas, col_alvo,
                         test_size=0.3, random_state=42):
    """
    Ajusta regressão logística e devolve coeficientes (log-odds) e desempenho.

    Parameters
    ----------
    df : DataFrame.
    colunas_explicativas : lista de colunas numéricas preditoras.
    col_alvo : coluna binária (0/1).
    test_size : fração para teste.

    Returns
    -------
    dict com coeficientes por variável, AUC e relatório de classificação.
    """
    X = df[colunas_explicativas].dropna()
    y = df.loc[X.index, col_alvo]

    if y.nunique() < 2:
        raise ValueError("A coluna-alvo precisa ter ambas as classes (0 e 1).")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )

    modelo = LogisticRegression(max_iter=1000)
    modelo.fit(X_train, y_train)

    y_proba = modelo.predict_proba(X_test)[:, 1]
    y_pred = modelo.predict(X_test)

    coeficientes = dict(zip(colunas_explicativas, modelo.coef_[0].round(4)))

    return {
        "coeficientes_logodds": coeficientes,
        "intercept": round(float(modelo.intercept_[0]), 4),
        "auc": round(float(roc_auc_score(y_test, y_proba)), 4),
        "relatorio": classification_report(y_test, y_pred, output_dict=True),
    }


# Exemplo de uso:
# df = pd.read_csv("02_base_analitica.csv")
# df["alto_crime"] = (df["taxa_homicidio"] > df["taxa_homicidio"].median()).astype(int)
# preditores = ["renda_media", "desigualdade", "escolaridade", "populacao"]
# res = correlacao_logistica(df, preditores, "alto_crime")
# print(res)